In [2]:
from __future__ import annotations

import time
import math
import requests
import numpy as np
import pandas as pd

from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from typing import Dict, Any, List, Optional, Tuple

from sklearn.impute import SimpleImputer
from sklearn.metrics import log_loss, accuracy_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression


# -----------------------------
# Config
# -----------------------------
ESPN_BASE = "https://site.web.api.espn.com/apis/site/v2/sports/basketball/nba"

DEFAULT_N_DAYS_HISTORY = 160  # NBA games are frequent; this is plenty
ROLL_WINDOW = 5              # last-5 rolling window
SLEEP_BETWEEN_CALLS = 0.20   # ESPN rate-limit friendliness

# Feature selection
MIN_NON_NULL_RATE = 0.25
DROP_CONSTANT_FEATURES = True

# -----------------------------
# Injuries (FREE - nbainjuries)
# -----------------------------
USE_INJURY_ADJUSTMENT = True
PRINT_UPCOMING_INJURY_TABLE = True

# Conservative log-odds penalties
INJURY_LOGIT_WEIGHTS = {
    "out_total": 0.055,
    "doubtful_total": 0.035,
    "questionable_total": 0.018,
    "probable_total": 0.008,
}


# -----------------------------
# Odds helpers
# -----------------------------
def to_american_odds(p: float) -> float:
    """Convert win probability p to fair American odds (no vig)."""
    p = float(p)
    p = min(max(p, 1e-6), 1 - 1e-6)
    if p >= 0.5:
        return -100.0 * p / (1.0 - p)
    return 100.0 * (1.0 - p) / p


def safe_float(x: Any) -> float:
    """Convert ESPN-ish values to float when possible; else NaN."""
    if x is None:
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    if isinstance(x, str):
        s = x.strip().replace("%", "")
        if s == "":
            return np.nan
        try:
            return float(s)
        except Exception:
            return np.nan
    if isinstance(x, dict):
        for k in ("value", "displayValue", "stat", "amount"):
            if k in x:
                return safe_float(x[k])
        return np.nan
    return np.nan


def sigmoid(x: float) -> float:
    return 1.0 / (1.0 + math.exp(-x))


def logit(p: float) -> float:
    p = float(p)
    p = min(max(p, 1e-6), 1 - 1e-6)
    return math.log(p / (1.0 - p))


# -----------------------------
# ESPN fetching
# -----------------------------
@dataclass
class ESPNClient:
    sleep: float = SLEEP_BETWEEN_CALLS

    def __post_init__(self):
        self.session = requests.Session()
        self.session.headers.update(
            {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"}
        )

    def get_scoreboard(self, date_yyyymmdd: Optional[str] = None) -> Optional[Dict[str, Any]]:
        url = f"{ESPN_BASE}/scoreboard"
        if date_yyyymmdd:
            url += f"?dates={date_yyyymmdd}"
        try:
            r = self.session.get(url, timeout=20)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f"[scoreboard] error for {date_yyyymmdd}: {e}")
            return None

    def get_summary(self, event_id: str) -> Optional[Dict[str, Any]]:
        url = f"{ESPN_BASE}/summary?event={event_id}"
        try:
            r = self.session.get(url, timeout=25)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f"[summary] error for event={event_id}: {e}")
            return None

    def get_teams_fullname_to_abbr(self) -> Dict[str, str]:
        """Map full team name -> abbreviation (needed for nbainjuries Team field mapping)."""
        url = f"{ESPN_BASE}/teams"
        try:
            r = self.session.get(url, timeout=20)
            r.raise_for_status()
            data = r.json()
        except Exception:
            return {}

        mapping: Dict[str, str] = {}
        try:
            teams = data["sports"][0]["leagues"][0]["teams"]
            for t in teams:
                team = t.get("team", {})
                name = str(team.get("displayName") or "").strip()
                abbr = str(team.get("abbreviation") or "").strip().upper()
                if name and abbr:
                    mapping[name] = abbr
        except Exception:
            pass

        return mapping


def daterange_yyyymmdd(days_back: int) -> List[str]:
    today_utc = datetime.now(timezone.utc).date()
    return [(today_utc - timedelta(days=i)).strftime("%Y%m%d") for i in range(days_back)]


def collect_events_by_dates(client: ESPNClient, days_back: int) -> List[Dict[str, Any]]:
    dates = daterange_yyyymmdd(days_back)
    events: List[Dict[str, Any]] = []

    for ds in dates:
        data = client.get_scoreboard(ds)
        if data and "events" in data:
            events.extend(data["events"])
        time.sleep(client.sleep)

    seen = set()
    uniq = []
    for e in events:
        eid = str(e.get("id"))
        if not eid or eid in seen:
            continue
        seen.add(eid)
        uniq.append(e)
    return uniq


def parse_scoreboard_event_minimal(event_obj: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    try:
        eid = str(event_obj["id"])
        date_str = event_obj.get("date")
        game_dt = pd.to_datetime(date_str).tz_convert(None) if date_str else pd.NaT

        competition = event_obj["competitions"][0]
        status = competition["status"]["type"]
        completed = bool(status.get("completed", False))

        competitors = competition["competitors"]
        if len(competitors) != 2:
            return None

        teams = []
        for c in competitors:
            team = c["team"]
            teams.append(
                {
                    "event_id": eid,
                    "game_date": game_dt,
                    "team_id": str(team.get("id")),
                    "team": team.get("displayName"),
                    "abbr": team.get("abbreviation"),
                    "home_away": c.get("homeAway"),
                    "score": safe_float(c.get("score")),
                    "winner": bool(c.get("winner", False)),
                    "completed": completed,
                }
            )
        return {"event_id": eid, "game_date": game_dt, "completed": completed, "teams": teams}
    except Exception:
        return None


def extract_team_stats_from_summary(summary_json: Dict[str, Any]) -> Dict[str, Dict[str, float]]:
    out: Dict[str, Dict[str, float]] = {}
    box = summary_json.get("boxscore", {}) if isinstance(summary_json, dict) else {}
    teams = box.get("teams", [])
    if not isinstance(teams, list):
        return out

    for t in teams:
        try:
            team_info = t.get("team", {})
            team_id = str(team_info.get("id"))
            if not team_id:
                continue

            stats_map: Dict[str, float] = {}
            stats_list = t.get("statistics", [])
            if isinstance(stats_list, list):
                for s in stats_list:
                    name = s.get("name") or s.get("abbreviation")
                    if not name:
                        continue
                    val = s.get("value")
                    if val is None:
                        val = s.get("displayValue")
                    stats_map[name] = safe_float(val)

            out[team_id] = stats_map
        except Exception:
            continue

    return out


def build_team_game_table(client: ESPNClient, events: List[Dict[str, Any]], keep_incomplete: bool = False) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []

    for ev in events:
        minimal = parse_scoreboard_event_minimal(ev)
        if not minimal:
            continue

        eid = minimal["event_id"]
        game_date = minimal["game_date"]
        completed = minimal["completed"]

        if (not keep_incomplete) and (not completed):
            continue

        summary = client.get_summary(eid) if completed else None
        team_stats = extract_team_stats_from_summary(summary) if summary else {}

        teams = minimal["teams"]
        if len(teams) != 2:
            continue

        a, b = teams[0], teams[1]

        for tm in teams:
            points = tm["score"]
            opp = b if tm["team_id"] == a["team_id"] else a
            opp_points = opp["score"]

            if completed and (pd.isna(points) or pd.isna(opp_points)):
                continue

            r = {
                "event_id": str(eid),
                "game_date": game_date,
                "team_id": tm["team_id"],
                "team": tm["team"],
                "abbr": tm["abbr"],
                "home_away": tm["home_away"],
                "completed": completed,
                "points": points,
                "opp_points": opp_points,
            }

            stats = team_stats.get(tm["team_id"], {})
            for k, v in stats.items():
                r[f"stat_{k}"] = v

            rows.append(r)

        time.sleep(client.sleep)

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")
    df["event_id"] = df["event_id"].astype(str)
    df["team_id"] = df["team_id"].astype(str)
    df = df.loc[:, ~df.columns.duplicated()].copy()

    for c in df.columns:
        if c.startswith("stat_") or c in ("points", "opp_points"):
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


# -----------------------------
# Dataset building
# -----------------------------
def make_matchup_dataset(team_game: pd.DataFrame, window: int = ROLL_WINDOW) -> Tuple[pd.DataFrame, List[str]]:
    tg = team_game.copy()
    tg = tg.dropna(subset=["event_id", "team_id", "game_date"]).copy()
    tg = tg.sort_values(["team_id", "game_date", "event_id"]).reset_index(drop=True)

    numeric_cols = [c for c in tg.columns if pd.api.types.is_numeric_dtype(tg[c])]
    exclude = {"points", "opp_points"}
    numeric_cols = [c for c in numeric_cols if c not in exclude]

    for c in numeric_cols + ["points", "opp_points"]:
        tg[f"roll_{c}_l{window}"] = (
            tg.groupby("team_id")[c].shift(1).rolling(window).mean()
        )

    base_cols = ["event_id", "game_date", "team_id", "team", "abbr", "home_away", "points"]
    g = tg[base_cols].copy()
    g = g.sort_values(["event_id", "home_away"])

    pairs = []
    for eid, grp in g.groupby("event_id"):
        if len(grp) != 2:
            continue

        home = grp[grp["home_away"] == "home"]
        away = grp[grp["home_away"] == "away"]
        if len(home) != 1 or len(away) != 1:
            continue

        home = home.iloc[0]
        away = away.iloc[0]

        pairs.append(
            {
                "event_id": str(eid),
                "game_date": home["game_date"],
                "home_team": home["team"],
                "away_team": away["team"],
                "home_id": home["team_id"],
                "away_id": away["team_id"],
                "pts_home": home["points"],
                "pts_away": away["points"],
                "y_home_win": int(home["points"] > away["points"])
                if (not pd.isna(home["points"]) and not pd.isna(away["points"]))
                else np.nan,
            }
        )

    games = pd.DataFrame(pairs)
    if games.empty:
        return games, []

    roll_cols = [c for c in tg.columns if c.startswith("roll_")]
    feats = tg[["event_id", "team_id"] + roll_cols].copy()

    df = games.merge(feats, left_on=["event_id", "home_id"], right_on=["event_id", "team_id"], how="left")
    df = df.drop(columns=["team_id"]).rename(columns={c: f"h_{c}" for c in roll_cols})

    df = df.merge(feats, left_on=["event_id", "away_id"], right_on=["event_id", "team_id"], how="left")
    df = df.drop(columns=["team_id"]).rename(columns={c: f"a_{c}" for c in roll_cols})

    feature_cols: List[str] = []
    for c in roll_cols:
        hc = f"h_{c}"
        ac = f"a_{c}"
        dc = f"d_{c}"
        if hc in df.columns and ac in df.columns:
            df[dc] = df[hc] - df[ac]
            feature_cols.append(dc)

    df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")
    df = df.dropna(subset=["game_date"]).sort_values("game_date").reset_index(drop=True)

    return df, feature_cols


# -----------------------------
# Training + calibration (FIXED)
# -----------------------------
def train_calibrated_model(df: pd.DataFrame, features: List[str]):
    df = df.copy()
    df = df.dropna(subset=["y_home_win"]).copy()
    df["y_home_win"] = df["y_home_win"].astype(int)

    for c in features:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    feats = [c for c in features if df[c].notna().any()]
    if DROP_CONSTANT_FEATURES:
        feats = [c for c in feats if df[c].nunique(dropna=True) > 1]

    if MIN_NON_NULL_RATE > 0:
        coverage = df[feats].notna().mean()
        feats = coverage[coverage >= MIN_NON_NULL_RATE].index.tolist()

    print(f"Features available: {len(features)}")
    print(f"Features used after cleanup/filter: {len(feats)}")

    df = df.sort_values("game_date").reset_index(drop=True)
    n = len(df)
    cut_train = int(n * 0.70)
    cut_cal = int(n * 0.85)

    train_df = df.iloc[:cut_train]
    cal_df = df.iloc[cut_train:cut_cal]
    test_df = df.iloc[cut_cal:]

    imp = SimpleImputer(strategy="median")
    X_train = imp.fit_transform(train_df[feats])
    y_train = train_df["y_home_win"].values

    X_cal = imp.transform(cal_df[feats])
    y_cal = cal_df["y_home_win"].values

    X_test = imp.transform(test_df[feats])
    y_test = test_df["y_home_win"].values

    base = LogisticRegression(
        solver="liblinear",
        max_iter=6000,
        C=0.7,
    )
    base.fit(X_train, y_train)

    # ✅ FIX: no cv="prefit" anymore
    X_traincal = np.vstack([X_train, X_cal])
    y_traincal = np.concatenate([y_train, y_cal])

    cal = CalibratedClassifierCV(base, method="sigmoid", cv=3)
    cal.fit(X_traincal, y_traincal)

    p_test = cal.predict_proba(X_test)[:, 1]
    y_hat = (p_test >= 0.5).astype(int)

    print("Test accuracy:", round(accuracy_score(y_test, y_hat), 4))
    print("Test logloss: ", round(log_loss(y_test, p_test), 4))

    return cal, imp, test_df, y_test, p_test, np.array(feats)


# -----------------------------
# Main
# -----------------------------
def main():
    client = ESPNClient(sleep=SLEEP_BETWEEN_CALLS)

    print("\n=== 1) Collect historical events ===")
    events_hist = collect_events_by_dates(client, days_back=DEFAULT_N_DAYS_HISTORY)
    print(f"Events collected (raw): {len(events_hist)}")

    print("\n=== 2) Build team-game table (completed games) ===")
    team_game = build_team_game_table(client, events_hist, keep_incomplete=False)
    if team_game.empty:
        print("No completed games found. Exiting.")
        return

    print("team_game shape:", team_game.shape)
    print("team_game date range:", team_game["game_date"].min(), "→", team_game["game_date"].max())

    num_cols = team_game.select_dtypes(include="number").columns.tolist()
    print("Numeric cols:", len(num_cols))
    print("Example numeric cols:", num_cols[:30])

    print("\n=== 3) Build matchup dataset + rolling features ===")
    df, FEATURE_COLS = make_matchup_dataset(team_game, window=ROLL_WINDOW)
    if df.empty or not FEATURE_COLS:
        print("Could not build matchup dataset / no features. Exiting.")
        return

    print("Matchups:", len(df))
    print("Feature candidates:", len(FEATURE_COLS))
    print("Feature example:", FEATURE_COLS[:10])

    print("\n=== 4) Train + calibrate model ===")
    model, imputer, test_df, y_test, p_test, FEATURES_USED = train_calibrated_model(df, FEATURE_COLS)

    out = test_df[["game_date", "home_team", "away_team", "pts_home", "pts_away", "y_home_win"]].copy()
    out["p_home_win"] = np.round(p_test, 3)
    out["fair_american_home"] = out["p_home_win"].apply(to_american_odds).round(1)
    out["fair_american_away"] = (1 - out["p_home_win"]).apply(to_american_odds).round(1)

    latest_day = out["game_date"].max()
    week_start = latest_day - pd.Timedelta(days=7)

    print("\n=== LATEST WEEK (COMPLETED, from TEST SPLIT) ===")
    print(
        out[out["game_date"] >= week_start]
        .sort_values("game_date", ascending=False)
        .head(30)
        .to_string(index=False)
    )

    print("\n=== Done ✅ ===")
    print("Features used:", len(FEATURES_USED))
    print("First 25:", FEATURES_USED[:25])


if __name__ == "__main__":
    main()



=== 1) Collect historical events ===
Events collected (raw): 744

=== 2) Build team-game table (completed games) ===
team_game shape: (1454, 34)
team_game date range: 2025-10-02 16:00:00 → 2026-01-22 03:00:00
Numeric cols: 27
Example numeric cols: ['points', 'opp_points', 'stat_fieldGoalsMade-fieldGoalsAttempted', 'stat_fieldGoalPct', 'stat_threePointFieldGoalsMade-threePointFieldGoalsAttempted', 'stat_threePointFieldGoalPct', 'stat_freeThrowsMade-freeThrowsAttempted', 'stat_freeThrowPct', 'stat_totalRebounds', 'stat_offensiveRebounds', 'stat_defensiveRebounds', 'stat_assists', 'stat_steals', 'stat_blocks', 'stat_turnovers', 'stat_teamTurnovers', 'stat_totalTurnovers', 'stat_technicalFouls', 'stat_totalTechnicalFouls', 'stat_flagrantFouls', 'stat_turnoverPoints', 'stat_fastBreakPoints', 'stat_pointsInPaint', 'stat_fouls', 'stat_largestLead', 'stat_leadChanges', 'stat_leadPercentage']

=== 3) Build matchup dataset + rolling features ===
Matchups: 727
Feature candidates: 28
Feature exam